# Notebook 04 — Backtesting
**Student 4 · Backtest Lead**

Runs all 7 portfolio strategies (S0–S6) through the `BacktestEngine`, produces
equity curves, drawdowns, weight charts, cost-sensitivity analysis, and saves
all results to `results/`.

**Prerequisites:** Run `03_models.ipynb` first.

In [ ]:
# ── Cell 1 · Setup & Load Data ────────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR    = '../data'
RAW_DIR     = os.path.join(DATA_DIR, 'raw')
PROC_DIR    = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load all inputs
predictions_df = pd.read_csv(os.path.join(RESULTS_DIR, 'predictions.csv'), parse_dates=['date'])
prices_df      = pd.read_csv(os.path.join(RAW_DIR, 'prices.csv'), index_col='date', parse_dates=True)
returns_df     = pd.read_csv(os.path.join(PROC_DIR, 'returns.csv'), index_col='date', parse_dates=True)
universe_df    = pd.read_csv(os.path.join(RAW_DIR, 'universe.csv'))
index_df       = pd.read_csv(os.path.join(RAW_DIR, 'index.csv'), index_col='date', parse_dates=True)

print(f'predictions_df : {predictions_df.shape}')
print(f'prices_df      : {prices_df.shape}')
print(f'returns_df     : {returns_df.shape}')
print(f'universe_df    : {universe_df.shape}')
print('Data loaded successfully.')

In [ ]:
# ── Cell 2 · Import BacktestEngine ────────────────────────────────────────────
from backtest import BacktestEngine

# Nifty returns for benchmark
nifty_returns = index_df['Return'] if 'Return' in index_df.columns else \
                index_df['Close'].pct_change()

engine = BacktestEngine(
    prices_df=prices_df,
    returns_df=returns_df,
    predictions_df=predictions_df,
    nifty_returns=nifty_returns,
    universe_df=universe_df,
    transaction_cost=0.002,   # 0.2% default
    weight_cap=0.15,
    lookback_cov=252,
)

print('BacktestEngine instantiated.')
print(f'Strategy IDs available: {engine.STRATEGY_IDS}')

In [ ]:
# ── Cell 3 · Run All 7 Strategies ─────────────────────────────────────────────
STRATEGY_LABELS = {
    'S0': 'NIFTY Benchmark',
    'S1': 'Equal Weight',
    'S2': 'Min Variance',
    'S3': 'Max Sharpe (Historical)',
    'S4': 'Max Sharpe (ML)',
    'S5': 'Top-5 ML Equal Weight',
    'S6': 'HRP',
}

results = {}
for sid in ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']:
    print(f'Running {sid} ({STRATEGY_LABELS[sid]}) …', end=' ')
    try:
        res = engine.run_strategy(sid)
        results[sid] = res
        print(f'CAGR={res["cagr"]*100:.1f}%  Sharpe={res["sharpe"]:.2f}')
    except Exception as e:
        print(f'ERROR: {e}')
        results[sid] = None

print('\nAll strategies complete.')

In [ ]:
# ── Cell 4 · Equity Curve Plot ────────────────────────────────────────────────
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
LINESTYLS = ['-', '--', '-.', ':', '-', '--', '-.']

fig, ax = plt.subplots(figsize=(16, 7))
for i, (sid, label) in enumerate(STRATEGY_LABELS.items()):
    if results.get(sid) is None:
        continue
    eq = results[sid]['equity_curve']
    lw = 2.5 if sid in ('S0', 'S4') else 1.5
    ax.plot(eq.index, eq.values, label=f'{sid}: {label}',
            color=COLORS[i], ls=LINESTYLS[i], lw=lw)

ax.set_title('Portfolio Equity Curves (Base = 1.0)', fontsize=14)
ax.set_ylabel('Portfolio Value')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'equity_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/equity_curves.png')

In [ ]:
# ── Cell 5 · Drawdown Plot ────────────────────────────────────────────────────
def compute_drawdown(equity_curve):
    roll_max = equity_curve.cummax()
    return (equity_curve - roll_max) / roll_max

fig, ax = plt.subplots(figsize=(16, 6))
for i, (sid, label) in enumerate(STRATEGY_LABELS.items()):
    if results.get(sid) is None:
        continue
    eq = results[sid]['equity_curve']
    dd = compute_drawdown(eq)
    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=COLORS[i])
    ax.plot(dd.index, dd.values, label=f'{sid}: {label}',
            color=COLORS[i], ls=LINESTYLS[i], lw=1.5)

ax.set_title('Drawdown Curves', fontsize=14)
ax.set_ylabel('Drawdown')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='lower left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'drawdowns.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/drawdowns.png')

In [ ]:
# ── Cell 6 · Weights Over Time (S4) ──────────────────────────────────────────
if results.get('S4') and 'weights' in results['S4']:
    weights_s4 = results['S4']['weights']
    fig, ax = plt.subplots(figsize=(16, 6))
    # Area chart of top 10 holdings by average weight
    avg_w = weights_s4.mean()
    top10 = avg_w.nlargest(10).index
    weights_s4[top10].plot.area(ax=ax, cmap='tab10', alpha=0.75)
    ax.set_title('S4 (Max Sharpe ML) — Portfolio Weights Over Time (Top 10 holdings)', fontsize=13)
    ax.set_ylabel('Weight')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'weights_s4.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/weights_s4.png')
else:
    print('S4 weights not available — skipping weight chart.')

In [ ]:
# ── Cell 7 · Strategy Metrics Table ──────────────────────────────────────────
def compute_metrics(res, label):
    if res is None:
        return None
    eq  = res['equity_curve']
    ret = eq.pct_change().dropna()

    n_years = (eq.index[-1] - eq.index[0]).days / 365.25
    cagr    = (eq.iloc[-1] / eq.iloc[0]) ** (1 / n_years) - 1 if n_years > 0 else np.nan
    vol     = ret.std() * np.sqrt(252)
    sharpe  = (ret.mean() * 252) / vol if vol > 0 else np.nan
    dd      = compute_drawdown(eq)
    max_dd  = dd.min()
    calmar  = cagr / abs(max_dd) if max_dd != 0 else np.nan

    avg_to  = res.get('avg_turnover', np.nan)

    return {
        'strategy': label,
        'cagr_pct': round(cagr * 100, 2),
        'vol_pct': round(vol * 100, 2),
        'sharpe': round(sharpe, 3),
        'max_dd_pct': round(max_dd * 100, 2),
        'calmar': round(calmar, 3),
        'avg_turnover': round(avg_to * 100, 2) if not np.isnan(avg_to) else np.nan,
    }

metrics_rows = []
for sid, label in STRATEGY_LABELS.items():
    m = compute_metrics(results.get(sid), f'{sid}: {label}')
    if m:
        metrics_rows.append(m)

strategy_metrics_df = pd.DataFrame(metrics_rows)
print('=== Strategy Performance Summary ===')
print(strategy_metrics_df.to_string(index=False))

In [ ]:
# ── Cell 8 · Cost Sensitivity Analysis ───────────────────────────────────────
COST_LEVELS = [0.001, 0.002, 0.003]  # 0.1%, 0.2%, 0.3%
cost_rows   = []

for cost in COST_LEVELS:
    for sid in ['S1', 'S4']:
        try:
            eng_c = BacktestEngine(
                prices_df=prices_df,
                returns_df=returns_df,
                predictions_df=predictions_df,
                nifty_returns=nifty_returns,
                universe_df=universe_df,
                transaction_cost=cost,
                weight_cap=0.15,
                lookback_cov=252,
            )
            res_c = eng_c.run_strategy(sid)
            eq    = res_c['equity_curve']
            ret   = eq.pct_change().dropna()
            n_y   = (eq.index[-1] - eq.index[0]).days / 365.25
            cagr  = (eq.iloc[-1] / eq.iloc[0]) ** (1 / n_y) - 1
            vol   = ret.std() * np.sqrt(252)
            sharpe= (ret.mean() * 252) / vol if vol > 0 else np.nan
            cost_rows.append({
                'strategy': sid, 'cost_bps': int(cost * 10000),
                'cagr_pct': round(cagr * 100, 2),
                'sharpe': round(sharpe, 3)
            })
            print(f'  {sid}  cost={cost*100:.1f}%  CAGR={cagr*100:.2f}%  Sharpe={sharpe:.3f}')
        except Exception as e:
            print(f'  {sid}  cost={cost*100:.1f}%  ERROR: {e}')

cost_df = pd.DataFrame(cost_rows)
print('\nCost Sensitivity:')
print(cost_df.to_string(index=False))

In [ ]:
# ── Cell 9 · Save Outputs ─────────────────────────────────────────────────────
# Collect all weights and daily returns
all_weights = []
all_daily   = []

for sid in STRATEGY_LABELS:
    if results.get(sid) is None:
        continue
    # Daily returns
    eq  = results[sid]['equity_curve']
    ret = eq.pct_change().dropna()
    ret_df = ret.to_frame(name='return')
    ret_df['strategy'] = sid
    all_daily.append(ret_df)

    # Weights
    if 'weights' in results[sid] and results[sid]['weights'] is not None:
        w_df = results[sid]['weights'].copy()
        w_df = w_df.stack().reset_index()
        w_df.columns = ['date', 'ticker', 'weight']
        w_df['strategy'] = sid
        all_weights.append(w_df)

daily_returns_df  = pd.concat(all_daily, ignore_index=False)
daily_returns_df.index.name = 'date'
daily_returns_df.reset_index(inplace=True)

if all_weights:
    weights_df = pd.concat(all_weights, ignore_index=True)
else:
    weights_df = pd.DataFrame(columns=['date','ticker','weight','strategy'])

daily_returns_df.to_csv(os.path.join(RESULTS_DIR, 'daily_returns.csv'), index=False)
weights_df.to_csv(os.path.join(RESULTS_DIR, 'weights.csv'), index=False)
strategy_metrics_df.to_csv(os.path.join(RESULTS_DIR, 'strategy_metrics.csv'), index=False)

print(f'Saved daily_returns.csv   {daily_returns_df.shape}')
print(f'Saved weights.csv         {weights_df.shape}')
print(f'Saved strategy_metrics.csv {strategy_metrics_df.shape}')
print('\n✅  Notebook 04 complete.')